# The problem with the perceptron: AND, OR and XOR

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.datasets import make_blobs, make_circles
from sklearn.linear_model import LogisticRegression, Perceptron

# the four possible rows of two binary inputs
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
tables = {"AND": np.array([0, 0, 0, 1]), "OR": np.array([0, 1, 1, 1]), "XOR": np.array([0, 1, 1, 0])}

## One perceptron per table

In [ ]:
for name, y in tables.items():
    p = Perceptron(random_state=0).fit(X, y)
    print(f"{name}: w = {p.coef_[0]}, b = {p.intercept_[0]}, accuracy = {p.score(X, y):.0%}, predictions = {p.predict(X)}")

## Decision regions

In [ ]:
# colour a grid of points by the predicted class, then draw the four rows on top
def regions(data, make_model, lim, size):
    fig = make_subplots(rows=1, cols=len(data), subplot_titles=list(data))
    gx, gy = np.meshgrid(np.linspace(*lim, 200), np.linspace(*lim, 200))
    for c, (name, (Xd, yd)) in enumerate(data.items(), start=1):
        m = make_model().fit(Xd, yd)
        zz = m.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
        fig.add_trace(go.Heatmap(x=gx[0], y=gy[:, 0], z=zz, zmin=0, zmax=1, showscale=False, opacity=0.25,
                                 colorscale=[[0, "#E45756"], [1, "#54A24B"]]), row=1, col=c)
        fig.add_trace(go.Scatter(x=Xd[:, 0], y=Xd[:, 1], mode="markers", showlegend=False,
                                 marker=dict(size=size, color=np.where(yd == 1, "#54A24B", "#E45756"))), row=1, col=c)
    return fig.update_layout(template="simple_white", height=400)

regions({k: (X, y) for k, y in tables.items()}, lambda: Perceptron(random_state=0), (-0.5, 1.5), 20)

## A TensorFlow Playground experiment, rebuilt: one sigmoid neuron (logistic regression)

In [ ]:
rng = np.random.default_rng(0)
Xx = rng.uniform(-5, 5, (400, 2))
Xb, yb = make_blobs(400, centers=[(-2.5, -2.5), (2.5, 2.5)], random_state=0)
Xc, yc = make_circles(400, noise=0.1, factor=0.4, random_state=0)
play = {"Two blobs": (Xb, yb),
        "XOR quadrants": (Xx, (Xx[:, 0] * Xx[:, 1] > 0).astype(int)),   # class 1 where x1 and x2 share a sign
        "Circles": (Xc * 4.5, yc)}
for name, (Xd, yd) in play.items():
    print(f"{name}: accuracy {LogisticRegression().fit(Xd, yd).score(Xd, yd):.0%}")
regions(play, LogisticRegression, (-6, 6), 6)

## Adding the feature x1 * x2 by hand makes XOR quadrants separable

In [ ]:
Xq, yq = play["XOR quadrants"]
Xq3 = np.c_[Xq, Xq[:, 0] * Xq[:, 1]]
print("with x1*x2 as a third input:", LogisticRegression().fit(Xq3, yq).score(Xq3, yq))

## XOR: the weights after each epoch
One epoch at a time (`warm_start=True`), to see where the weights are after each pass.

In [ ]:
import warnings
warnings.filterwarnings("ignore")          # one-epoch fits warn that they did not converge
y_xor = tables["XOR"]
p = Perceptron(random_state=0, max_iter=1, tol=None, warm_start=True)
for epoch in range(1, 7):
    p.fit(X, y_xor)
    print(epoch, p.coef_[0], p.intercept_[0])
# the four updates y_i * (x_i, 1), with labels -1/+1, add up to zero
y_pm = np.where(y_xor == 1, 1, -1)
print((y_pm[:, None] * np.c_[X, np.ones(4)]).sum(axis=0))
